# Economic Simulation & Battery Steering

**ML6 Imbalance Forecasting Workshop — Team SIAL**

This notebook translates System Imbalance forecasts into a simple battery
steering strategy and evaluates the resulting BRP settlement impact.

The focus is deliberately different from the core forecasting notebook:
instead of improving predictive accuracy, the objective is to connect
forecasts to an operational decision.

The workflow includes:

- minute-level battery steering;
- State of Charge constraints;
- charging/discharging efficiency;
- 15-minute imbalance settlement;
- comparison against a no-action baseline;
- cumulative economic impact.

> The original workshop datasets are not redistributed in this portfolio.
> This notebook expects the original parquet files in a local `data/`
> directory and, optionally, forecasts exported by Notebook 01.

## 1. Decision logic

The strategy uses the predicted System Imbalance direction:

- **forecast SI < 0** → system expected to be short → discharge the battery;
- **forecast SI > 0** → system expected to be long → charge the battery;
- battery actions are constrained by the current State of Charge (SOC).

The battery is therefore used as a flexible asset that attempts to move in a
direction that is beneficial to the system.

In [ ]:
from pathlib import Path
from typing import Dict, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 2. Load workshop data

The original ML6 data are not included in this repository.

Place the original parquet files inside:

```text
data/
```

The helper below loads them into a dictionary keyed by filename.

In [ ]:
def load_parquet_folder_as_dict(
    folder: Union[str, Path],
    pattern: str = "*.parquet",
    index_col: Optional[str] = None,
    to_datetime_index: bool = True,
) -> Dict[str, pd.DataFrame]:
    folder = Path(folder)

    if not folder.exists():
        raise FileNotFoundError(
            "Missing `data/` directory. Add the original ML6 workshop "
            "parquet files locally before running this notebook."
        )

    result = {}

    for path in sorted(folder.glob(pattern)):
        df = pd.read_parquet(path)

        if index_col is not None and index_col in df.columns:
            df = df.set_index(index_col)

        if to_datetime_index:
            try:
                df.index = pd.to_datetime(df.index, utc=True)
            except Exception:
                pass

        result[path.stem] = df

    return result


df_dict = load_parquet_folder_as_dict("data")

print(f"Loaded {len(df_dict)} parquet datasets.")

## 3. Load forecasts from the core model

Notebook 01 can export a `predictions.parquet` file containing one System
Imbalance forecast per minute.

This notebook uses those predictions when the file is available.

A delayed-SI fallback can be used for demonstration, but its economic result
must not be interpreted as the performance of the final forecasting model.

In [ ]:
forecast_path = Path("predictions.parquet")

forecast_series = None

if forecast_path.exists():
    forecast_df = pd.read_parquet(forecast_path)

    if "prediction" not in forecast_df.columns:
        raise ValueError(
            "`predictions.parquet` must contain a `prediction` column."
        )

    forecast_df.index = pd.to_datetime(forecast_df.index, utc=True)
    forecast_series = forecast_df["prediction"]

    print(f"Loaded {len(forecast_series):,} model forecasts.")
else:
    print(
        "No `predictions.parquet` found. "
        "The simulation can still run with the explicitly labelled "
        "delayed-SI fallback."
    )

## 4. Battery steering policy

The control rule is intentionally simple so that the relationship between the
forecast and the operational decision remains transparent.

A more advanced strategy could additionally use forecast magnitude,
imbalance-price forecasts, degradation costs and probabilistic uncertainty.

In [ ]:
def trading_strategy(
    forecast_si_mw: float,
    soc_mwh: float,
    battery,
) -> float:
    """Return a normalized charge/discharge request in [-1, 1].

    Convention
    ----------
    forecast_si_mw < 0:
        expected system shortage -> discharge -> inject into grid.

    forecast_si_mw > 0:
        expected system surplus -> charge -> consume from grid.

    Battery control
    ---------------
    positive action:
        charge request.

    negative action:
        discharge request.
    """
    soc_fraction = soc_mwh / battery.capacity_mwh

    if forecast_si_mw < 0 and soc_fraction > 0.10:
        return -1.0

    if forecast_si_mw > 0 and soc_fraction < 0.90:
        return 1.0

    return 0.0

## 5. Battery model

The asset model includes:

- maximum power;
- energy capacity;
- round-trip efficiency approximation;
- minimum SOC;
- physical clipping of infeasible actions.

In [ ]:
class Battery:
    """Simple battery model for minute-level steering."""

    def __init__(
        self,
        power_mw: float = 1.0,
        capacity_mwh: float = 2.0,
        efficiency: float = 0.95,
        soc_init_fraction: float = 0.50,
        min_soc_fraction: float = 0.20,
    ) -> None:
        self.power_mw = float(power_mw)
        self.capacity_mwh = float(capacity_mwh)
        self.efficiency = float(efficiency)

        self.soc_mwh = (
            float(soc_init_fraction)
            * self.capacity_mwh
        )

        self.min_soc_mwh = (
            float(min_soc_fraction)
            * self.capacity_mwh
        )

    def act(
        self,
        power_fraction: float,
        dt_hours: float,
    ) -> float:
        """Apply an action and return realized grid energy flow in MWh.

        Positive grid flow:
            battery discharges and injects energy.

        Negative grid flow:
            battery charges and consumes energy.
        """
        u = float(np.clip(power_fraction, -1.0, 1.0))
        eta = self.efficiency

        requested_grid_mwh = (
            -u * self.power_mw * dt_hours
        )

        max_charge_grid_mwh = (
            max(
                0.0,
                self.capacity_mwh - self.soc_mwh,
            )
            / eta
        )

        max_discharge_grid_mwh = (
            max(
                0.0,
                self.soc_mwh - self.min_soc_mwh,
            )
            * eta
        )

        realized_grid_mwh = float(
            np.clip(
                requested_grid_mwh,
                -max_charge_grid_mwh,
                max_discharge_grid_mwh,
            )
        )

        if realized_grid_mwh >= 0.0:
            self.soc_mwh -= realized_grid_mwh / eta
        else:
            self.soc_mwh += (-realized_grid_mwh) * eta

        self.soc_mwh = float(
            np.clip(
                self.soc_mwh,
                self.min_soc_mwh,
                self.capacity_mwh,
            )
        )

        return realized_grid_mwh

## 6. Build minute-level simulation inputs

The simulation combines:

- realized System Imbalance;
- the model forecast;
- 15-minute imbalance settlement prices;
- a reproducible synthetic BRP imbalance.

If no external forecast is provided, the code can use delayed realized SI as
an **illustrative fallback only**.

In [ ]:
def prepare_simulation_data(
    df_dict: dict,
    start: str = "2026-02-01 00:00:00",
    end: str = "2026-03-01 00:00:00",
    forecast_series: pd.Series | None = None,
    forecast_delay_min: int = 2,
    brp_sigma_mw: float = 2.0,
    brp_seed: int = 42,
) -> pd.DataFrame:

    si_1min = (
        df_dict["df_hist_1_min_imbalance_prices"]
        ["systemimbalance"]
    )

    price_15min = (
        df_dict["df_hist_15_min_imbalance_prices"]
        ["imbalanceprice"]
    )

    start_ts = pd.Timestamp(start, tz="UTC")
    end_ts = pd.Timestamp(end, tz="UTC")

    minute_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="1min",
        inclusive="left",
    )

    qh_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="15min",
        inclusive="left",
    )

    data = pd.DataFrame(index=minute_index)

    data["actual_si_mw"] = (
        si_1min.reindex(minute_index)
    )

    if forecast_series is None:
        data["forecast_si_mw"] = (
            si_1min
            .shift(forecast_delay_min)
            .reindex(minute_index)
        )

        data.attrs["forecast_source"] = (
            "delayed-SI fallback"
        )
    else:
        data["forecast_si_mw"] = (
            forecast_series.reindex(minute_index)
        )

        data.attrs["forecast_source"] = (
            "external model forecasts"
        )

    qh_price = (
        price_15min
        .reindex(qh_index)
        .ffill()
        .bfill()
    )

    data["imbalance_price_eur_per_mwh"] = (
        data.index.floor("15min").map(qh_price)
    )

    data = data.dropna(
        subset=[
            "actual_si_mw",
            "imbalance_price_eur_per_mwh",
        ]
    ).copy()

    data["forecast_si_mw"] = (
        data["forecast_si_mw"]
        .fillna(0.0)
    )

    rng = np.random.default_rng(brp_seed)

    data["brp_inherent_imbalance_mw"] = (
        rng.normal(
            0.0,
            brp_sigma_mw,
            len(data),
        )
    )

    return data

## 7. Settlement simulation

The battery acts every minute, while financial settlement is aggregated to
15-minute periods.

The no-action baseline and steered case use the same underlying synthetic BRP
imbalance so that the comparison isolates the impact of the battery strategy.

In [ ]:
def run_case(
    minute_data: pd.DataFrame,
    battery: Battery,
    strategy_fn=None,
    minute_dt: float = 1.0 / 60.0,
) -> dict:

    asset_flow_mwh = []
    soc_series = []
    actions = []

    for _, row in minute_data.iterrows():

        if strategy_fn is None:
            action = 0.0
        else:
            action = float(
                strategy_fn(
                    row["forecast_si_mw"],
                    battery.soc_mwh,
                    battery,
                )
            )

        realized_flow = battery.act(
            action,
            minute_dt,
        )

        actions.append(action)
        asset_flow_mwh.append(realized_flow)
        soc_series.append(battery.soc_mwh)

    results = minute_data.copy()

    results["action"] = actions
    results["asset_flow_mwh"] = asset_flow_mwh
    results["soc_mwh"] = soc_series

    results["brp_inherent_imbalance_mwh"] = (
        results["brp_inherent_imbalance_mw"]
        * minute_dt
    )

    results["net_imbalance_mwh"] = (
        results["brp_inherent_imbalance_mwh"]
        + results["asset_flow_mwh"]
    )

    settlement = pd.DataFrame(
        index=results.resample("15min").first().index
    )

    settlement["brp_inherent_imbalance_mwh"] = (
        results["brp_inherent_imbalance_mwh"]
        .resample("15min")
        .sum()
    )

    settlement["asset_flow_mwh"] = (
        results["asset_flow_mwh"]
        .resample("15min")
        .sum()
    )

    settlement["final_imbalance_mwh"] = (
        results["net_imbalance_mwh"]
        .resample("15min")
        .sum()
    )

    settlement["avg_price_eur_per_mwh"] = (
        results["imbalance_price_eur_per_mwh"]
        .resample("15min")
        .mean()
    )

    settlement["settlement_eur"] = (
        settlement["final_imbalance_mwh"]
        * settlement["avg_price_eur_per_mwh"]
    )

    settlement["cumulative_settlement_eur"] = (
        settlement["settlement_eur"]
        .cumsum()
    )

    return {
        "minute_df": results,
        "settlement_df": settlement,
        "total_settlement_eur": (
            settlement["settlement_eur"].sum()
        ),
        "soc_series_mwh": (
            results["soc_mwh"]
            .resample("15min")
            .last()
        ),
    }

In [ ]:
def run_brp_imbalance_simulation(
    df_dict: dict,
    forecast_series: pd.Series | None = None,
    start: str = "2026-02-01 00:00:00",
    end: str = "2026-03-01 00:00:00",
    forecast_delay_min: int = 2,
    brp_sigma_mw: float = 2.0,
    brp_seed: int = 42,
) -> dict:

    minute_data = prepare_simulation_data(
        df_dict=df_dict,
        start=start,
        end=end,
        forecast_series=forecast_series,
        forecast_delay_min=forecast_delay_min,
        brp_sigma_mw=brp_sigma_mw,
        brp_seed=brp_seed,
    )

    minute_dt = 1.0 / 60.0

    baseline_imbalance_mwh = (
        minute_data["brp_inherent_imbalance_mw"]
        * minute_dt
    ).resample("15min").sum()

    baseline_price = (
        minute_data["imbalance_price_eur_per_mwh"]
        .resample("15min")
        .mean()
    )

    baseline_settlement = (
        baseline_imbalance_mwh
        * baseline_price
    )

    baseline = {
        "total_settlement_eur": (
            baseline_settlement.sum()
        ),
        "cumulative_settlement_eur": (
            baseline_settlement.cumsum()
        ),
    }

    steered = run_case(
        minute_data=minute_data,
        battery=Battery(),
        strategy_fn=trading_strategy,
        minute_dt=minute_dt,
    )

    return {
        "forecast_source": (
            minute_data.attrs.get(
                "forecast_source",
                "unknown",
            )
        ),
        "baseline": baseline,
        "steered": steered,
    }

## 8. Run the simulation

If `predictions.parquet` exists, the simulation uses the forecasts generated
by the core model. Otherwise, it runs with the clearly labelled delayed-SI
fallback.

This distinction matters: economic results from the fallback must not be
reported as model performance.

In [ ]:
simulation_results = run_brp_imbalance_simulation(
    df_dict=df_dict,
    forecast_series=forecast_series,
    brp_sigma_mw=2.0,
)

baseline = simulation_results["baseline"]
steered = simulation_results["steered"]

baseline_total = float(
    baseline["total_settlement_eur"]
)

steered_total = float(
    steered["total_settlement_eur"]
)

value_add = (
    steered_total - baseline_total
)

print(
    "Forecast source:",
    simulation_results["forecast_source"],
)
print(
    f"Baseline settlement: {baseline_total:,.2f} EUR"
)
print(
    f"Steered settlement:  {steered_total:,.2f} EUR"
)
print(
    f"Incremental impact:  {value_add:,.2f} EUR"
)

## 9. Economic and operational diagnostics

In [ ]:
baseline_curve = (
    baseline["cumulative_settlement_eur"]
)

steered_curve = (
    steered["settlement_df"]
    ["cumulative_settlement_eur"]
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    baseline_curve.index,
    baseline_curve,
    label="Baseline — no battery action",
    linestyle="--",
)

ax.plot(
    steered_curve.index,
    steered_curve,
    label="Forecast-driven battery steering",
)

ax.set_title(
    "Cumulative BRP Settlement Impact"
)

ax.set_ylabel("EUR")
ax.set_xlabel("Time")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

soc = steered["soc_series_mwh"]

ax.plot(
    soc.index,
    soc,
    label="Battery SOC",
)

ax.set_title("Battery State of Charge")
ax.set_ylabel("MWh")
ax.set_xlabel("Time")
ax.legend()

plt.tight_layout()
plt.show()

## 10. Interpretation

This simulation illustrates an important point from the workshop:

**forecasting quality is valuable only when it improves a downstream
decision.**

A model with slightly worse average RMSE can still be economically useful if
it better identifies the sign and magnitude of high-impact imbalance events.

The simple strategy implemented here is intentionally transparent. It does not
claim to be an optimal battery-dispatch policy.

## Limitations

The simulation makes several simplifying assumptions:

- the BRP's inherent imbalance is synthetic;
- battery degradation costs are not modelled;
- the steering policy uses the sign of predicted SI rather than an optimized
  decision threshold;
- future prices are not explicitly forecast;
- the asset is represented by a simplified efficiency/SOC model;
- the workshop settlement simulation is an educational approximation rather
  than a production market engine.

These limitations are important when interpreting any simulated economic
result.

## Key Takeaways

This notebook connects machine-learning forecasts to an operational energy
decision.

It demonstrates:

- forecast-driven asset steering;
- physical battery constraints;
- minute-level control and quarter-hour settlement;
- comparison against a no-action baseline;
- the distinction between statistical accuracy and economic value.

Together with the core forecasting notebook, it completes the project's
end-to-end path from **real-time data → prediction → decision → economic
evaluation**.